# rotation-matrix-3d-y-axis — ex6: rotation sweep — visualize 5 angles

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d-y-axis`. Running the final beacon cell reports progress against the `Numpy: Applied patterns and advanced` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rotation-matrix-3d-y-axis`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d-y-axis"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Y-axis rotation — quick refresher

**The matrix.** Right-hand rotation by `θ` about Y:
```
R_y(θ) = [[ cos θ,  0,  sin θ],
          [ 0,      1,  0    ],
          [-sin θ,  0,  cos θ]]
```
Anything along Y stays put (middle row `[0,1,0]`); the X-Z plane rotates.

**Acting on data.** Column-vector form: `v' = R @ v`. Batch of row-vectors `(N, 3)`: `points' = points @ R.T`. Composition: `R(α) @ R(β) = R(α + β)` for single-axis rotations; multi-axis rotations don't commute.

**Numerical truth.** Rotation matrices are orthogonal: `R @ R.T = I` and `R.inverse() == R.T`. Floating-point composition accumulates ~1e-7 error per matmul.

### Exercise 6 — rotation sweep — visualize 5 angles

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply Y-rotation across a sweep of angles to a single point cloud and render each rotated snapshot as a subplot to inspect the trajectory visually.
> Keywords: rotation, batch, visualization, matplotlib, sweep
> ```

**KCs targeted:** `rotation-matrix-y-construct`, `rotate-batch-of-points`

Implement `ex6_rotation_sweep(points, angles)`. Given a `(N, 3)` batch of points and a 1-D tensor of `M` scalar angles (in radians), return a stacked tensor of shape `(M, N, 3)` where row `m` is `points` rotated by `angles[m]` around the Y axis.

**Hint.** Build `R_y(θ_m)` once per angle in a Python loop, then `points @ R.T`. Stack the per-angle results with `t.stack(..., dim=0)`. (There's also a fully vectorized version using `t.stack` of the rotation matrices first; the loop version is fine for this exercise.)

The test verifies values at the canonical angles 0, π/2, π, then renders 5 X-Z-plane subplots showing the rotated point cloud at each angle so you can see the trajectory.

In [ ]:
def ex6_rotation_sweep(points: Tensor, angles: Tensor) -> Tensor:
    """Rotate (N, 3) `points` by each scalar in `angles`. Returns (M, N, 3)."""
    raise NotImplementedError()


def _test_ex6():
    import math

    # A simple 4-point square in the X-Z plane (Y=0).
    points = t.tensor([
        [ 1.0, 0.0,  0.0],
        [ 0.0, 0.0,  1.0],
        [-1.0, 0.0,  0.0],
        [ 0.0, 0.0, -1.0],
    ])
    angles = t.tensor([0.0, math.pi / 6, math.pi / 3, math.pi / 2, math.pi])
    out = ex6_rotation_sweep(points, angles)

    # Shape check.
    assert out.shape == (5, 4, 3), f'expected (5, 4, 3), got {tuple(out.shape)}'

    # Angle 0 must be the identity.
    assert t.allclose(out[0], points, atol=1e-6), 'angle 0 should leave points unchanged'

    # Angle π should negate the X-Z components, leave Y alone.
    expected_pi = points.clone()
    expected_pi[:, 0] = -points[:, 0]
    expected_pi[:, 2] = -points[:, 2]
    assert t.allclose(out[4], expected_pi, atol=1e-5), 'angle π should negate X and Z'

    # Angle π/2: +X → -Z, +Z → +X.
    p_x_axis = out[3, 0]
    assert t.allclose(p_x_axis, t.tensor([0.0, 0.0, -1.0]), atol=1e-6), \
        f'+X at π/2 should be (0,0,-1), got {p_x_axis}'
    p_z_axis = out[3, 1]
    assert t.allclose(p_z_axis, t.tensor([1.0, 0.0, 0.0]), atol=1e-6), \
        f'+Z at π/2 should be (1,0,0), got {p_z_axis}'

    # Length preservation across the whole sweep.
    norms = out.pow(2).sum(dim=-1).sqrt()
    assert t.allclose(norms, t.ones(5, 4), atol=1e-5), 'all rotated points should have unit norm'

    # Visualize: 5 subplots showing the X-Z projection at each angle.
    fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
    for i, ax in enumerate(axes):
        pts = out[i]
        ax.scatter(pts[:, 0].numpy(), pts[:, 2].numpy(), s=60, c=range(4), cmap='viridis')
        ax.scatter(points[:, 0].numpy(), points[:, 2].numpy(), s=30, c='lightgray', marker='x')
        ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
        ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
        ax.set_title(f'θ = {angles[i].item():.2f} rad')
        ax.set_xlabel('X'); ax.set_ylabel('Z')
    fig.suptitle('Y-axis rotation sweep (gray X = original)')
    fig.tight_layout()
    plt.show()
    print(f'output shape: {tuple(out.shape)}')
    print(f'norms per angle: {norms.mean(dim=-1).tolist()} (all should be ~1.0)')
    _dd_passed.add('ex6')
    print("ex6 ✓")

_test_ex6()

<details><summary>Solution</summary>

```python
def ex6_rotation_sweep(points: Tensor, angles: Tensor) -> Tensor:
    snapshots = []
    for theta in angles:
        c, s = t.cos(theta).item(), t.sin(theta).item()
        R = t.tensor([
            [c,   0.0, s  ],
            [0.0, 1.0, 0.0],
            [-s,  0.0, c  ],
        ])
        snapshots.append(points @ R.T)
    return t.stack(snapshots, dim=0)
```

**Loop vs vectorized.** The loop version is clearest for first contact. For performance you can build a `(M, 3, 3)` stack of rotation matrices in one go with `t.stack` of `(M,)` cos/sin tensors, then do a single batched matmul `points @ Rs.transpose(-1, -2)` (broadcast against the `M` axis). The math is identical; only the wall-clock differs.

**Where you'll use this.** Generating training-time augmentations (random rotations as a `(M, ...)` axis), orbiting a camera around a fixed scene, rendering a turntable GIF of a 3-D model. Anywhere you need *the same data, different viewing angles*.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()